In [1]:
import json
from pathlib import Path
from typing import Dict, Any
from src.backend.db.dbFacade import DBFacade
from openai import OpenAI
from src.backend.modules.adviser_pipeline import AdvisorProcessor
from src.backend.utils.configs import Config
from src.backend.models.db_models import *
from src.backend.db.dbFacade import DBFacade
import asyncio
 
 
 


In [2]:
db = DBFacade()
await db.create_tables()

[2025-10-06 20:23:01] [INFO] Tables created successfully!


In [3]:
products = await db.get_company_products_by_consultant_id('68f997c0-95d5-4c88-b0c6-c5c13061ec2a')

In [4]:
print(products)

[ProductResponse(id='07549e01-ec9b-4c09-bb80-3f9c1faac5e7', company_id='23f120cf-f799-4d19-981c-fca0e031d6a9', product_name='Sep Ira', product_code='SEP-IRA', product_type='Pension Plans', description='A SEP-IRA (Simplified Employee Pension Individual Retirement Account) is a defined contribution plan (defined_contribution) for self-employed individuals (self-employed) and small business owners (private_employer). It offers a simple, low-cost way for employers to make retirement contributions for themselves and their employees. Participation is mandatory for all eligible employees age 21+ with at least $750 in compensation and having worked in 3 of the last 5 years (employee_age_requirement...'), ProductResponse(id='0d79a798-f0f3-4c48-a09a-86e8905dadb6', company_id='23f120cf-f799-4d19-981c-fca0e031d6a9', product_name='Annuities', product_code='ANNUITIES', product_type='Investment Plans', description='Annuities are insurance contracts (insurance_company_owned) that provide guaranteed pa

In [5]:
 
profile = await db.get_client_full_info('1d4ca5fb-1190-4c2a-b3dd-ecaf79380dc7')



In [6]:
profile

{'client': ClientResponse(id='1d4ca5fb-1190-4c2a-b3dd-ecaf79380dc7', citizenship='U.S. citizen', marital_status='single', id_number='TX9876543', id_type="Driver's License", country_of_issuance='United States', id_issuance_date=datetime.datetime(2022, 7, 1, 0, 0), id_expiration_date=datetime.datetime(2030, 6, 12, 0, 0)),
 'persons': [personResponse(id='942b0fe0-0c8e-4833-83d7-3352d28e0c19', client_id='1d4ca5fb-1190-4c2a-b3dd-ecaf79380dc7', beneficiary_id='53e29eea-ef1a-45d6-a032-83d0a2c43ca3', first_name='Mary', middle_name='Elizabeth', last_name='Collins', date_of_birth=datetime.datetime(1990, 6, 12, 0, 0), sex='Female', ssn_or_tin='456-78-9123', email='mary.collins.rn@gmail.com', phone_number='+1-512-555-2847', phone_alt='+1-512-555-2848'),
  personResponse(id='d2f2d481-952d-4ede-a653-e4a57085d24b', client_id='1d4ca5fb-1190-4c2a-b3dd-ecaf79380dc7', beneficiary_id='53e29eea-ef1a-45d6-a032-83d0a2c43ca3', first_name='Emma', middle_name='Rose', last_name='Collins', date_of_birth=datetime.

In [9]:
def load_transcript(recording_path: str) -> str:
    """Load meeting transcript."""
    with open(Path(recording_path) / "full_final_transcript.txt", 'r', encoding='utf-8') as f:
        return f.read()
transcript = load_transcript("recordings/full/nzz-tevk-zab_2025-09-15_18-55-07")
print(f"✓ Loaded transcript ({len(transcript)} characters)")

✓ Loaded transcript (8256 characters)


In [10]:
unified_data = {
    "products": products,
    "transcript": transcript,
    "selected_profile": profile 
}

print("✓ Unified data prepared")

✓ Unified data prepared


In [11]:
print(unified_data)

{'products': [ProductResponse(id='07549e01-ec9b-4c09-bb80-3f9c1faac5e7', company_id='23f120cf-f799-4d19-981c-fca0e031d6a9', product_name='Sep Ira', product_code='SEP-IRA', product_type='Pension Plans', description='A SEP-IRA (Simplified Employee Pension Individual Retirement Account) is a defined contribution plan (defined_contribution) for self-employed individuals (self-employed) and small business owners (private_employer). It offers a simple, low-cost way for employers to make retirement contributions for themselves and their employees. Participation is mandatory for all eligible employees age 21+ with at least $750 in compensation and having worked in 3 of the last 5 years (employee_age_requirement...'), ProductResponse(id='0d79a798-f0f3-4c48-a09a-86e8905dadb6', company_id='23f120cf-f799-4d19-981c-fca0e031d6a9', product_name='Annuities', product_code='ANNUITIES', product_type='Investment Plans', description='Annuities are insurance contracts (insurance_company_owned) that provide 

In [12]:
print(products)

[ProductResponse(id='07549e01-ec9b-4c09-bb80-3f9c1faac5e7', company_id='23f120cf-f799-4d19-981c-fca0e031d6a9', product_name='Sep Ira', product_code='SEP-IRA', product_type='Pension Plans', description='A SEP-IRA (Simplified Employee Pension Individual Retirement Account) is a defined contribution plan (defined_contribution) for self-employed individuals (self-employed) and small business owners (private_employer). It offers a simple, low-cost way for employers to make retirement contributions for themselves and their employees. Participation is mandatory for all eligible employees age 21+ with at least $750 in compensation and having worked in 3 of the last 5 years (employee_age_requirement...'), ProductResponse(id='0d79a798-f0f3-4c48-a09a-86e8905dadb6', company_id='23f120cf-f799-4d19-981c-fca0e031d6a9', product_name='Annuities', product_code='ANNUITIES', product_type='Investment Plans', description='Annuities are insurance contracts (insurance_company_owned) that provide guaranteed pa

In [13]:

processor = AdvisorProcessor()
result = await processor.process_advice(unified_data)

[2025-10-06 20:24:22] [INFO] Strategy analysis completed
[2025-10-06 20:24:38] [INFO] Meeting analysis completed
[2025-10-06 20:24:39] [INFO] Generated 10 tags
[2025-10-06 20:24:39] [INFO] Analysis completed successfully


In [14]:
from IPython.display import Markdown, display

 
print(result.summary)
print(result.tags)

# COMPREHENSIVE FINANCIAL ADVISOR REPORT

## MEETING ANALYSIS
# CLIENT PROFILE SUMMARY

**Demographics:**  
- Age: 48 (turning 48 this year)  
- Family status: Married (wife: Sarah), two children (Emily and Michael Jr.)  
- Employment: Senior VP at Goldman Sachs  
- Income level: $550,000 base salary plus bonuses (high net worth)

**Financial Position:**  
- Assets: Significant 401(k) and IRA holdings, Goldman Sachs stock options  
- Investments: Heavy exposure to equities (via employer stock and traditional investments)  
- Cash flow: Strong, high earning capacity  
- Debt: Not specified (assumed manageable given position and profile)

**Goals:**  
- Primary objectives:  
  - Diversify investment portfolio beyond current 401(k) and IRA  
  - Increase life insurance coverage for estate planning and family protection  
  - Reduce concentration risk in equities  
- Timelines:  
  - Immediate action on insurance and managed futures fund  
  - Ongoing review for further diversification  
-

In [ ]:
 
from src.backend.modules.meetingAnalizer import MeetingAnalizer

async def main():
    analizer = MeetingAnalizer()

    # Пример текста, который ты хочешь проверить
    transcript_text = """
        Michael Chen: Good morning, Ethan. This is Rachel from WealthGuard Financial. How are you today?Ethan: Hi Rachel, I’m doing well, thanks. How about you?Michael Chen: Doing great, thanks for asking. The reason for my call today is to offer a complimentary review of your financial portfolio. We’ve identified a few potential options 401k that might help you diversify your investments and secure your retirement plans. Would it be okay if I go over them briefly?Ethan: Sure, go ahead.Michael Chen: Excellent. The first option I wanted to mention is a low-risk municipal bond fund. It offers tax-free interest income and is designed to be conservative for long-term stability.Ethan: Hmm, I appreciate that, but I’m already maxing out my 401k contributions through Fidelity, and I prefer to keep my taxable investments flexible. I’m not really looking for tax-free municipal bonds at the moment.Michael Chen: Got it, that makes sense. The second option is a variable life insurance policy. It combines life coverage with investment growth potential, which could supplement your retirement and provide additional protection for Lucas.Ethan: I see. Honestly, I already have adequate life insurance coverage, and I don’t want to tie up additional funds in a policy that mixes investments with insurance. I prefer keeping them separate.Michael Chen: Understood. The third option is a target-date mutual fund aligned with your projected retirement year. It automatically adjusts risk allocation as you approach retirement.Ethan: I actually like target-date funds in theory, but my current 401k already offers a professionally managed fund that does exactly that. I don’t need another one right now.Michael Chen: Makes sense, Ethan. The fourth idea I had was a high-yield corporate bond ETF, which provides higher returns than traditional bonds, but with slightly more risk.Ethan: I’m cautious about corporate bonds at the moment. The market volatility is a bit too high for my taste, especially given that I want to keep retirement funds stable.Michael Chen: I completely understand. Lastly, I thought about a Roth IRA top-up strategy, considering your contributions. This would allow for tax-free growth beyond your 401k limits.Ethan: I appreciate the suggestion, but I already contribute to a Roth component through my 401k, and I’m comfortable with my current allocation. I don’t want to make additional contributions right now.Michael Chen: Thank you for clarifying, Ethan. It sounds like you’re confident with your current retirement setup. Just to summarize, we went over municipal bonds, variable life insurance, a target-date fund, corporate bond ETF, and Roth IRA top-ups, and none of these fit your current goals, correct?Ethan: That’s correct. I’m happy with the current setup and don’t want to make changes.Michael Chen: Understood. I appreciate your time and honesty. If anything changes in the future or if you want to revisit these options, I’d be happy to schedule another conversation.Ethan: Thanks, Rachel. I’ll keep that in mind.Michael Chen: Great, Ethan. Enjoy the rest of your day!Ethan: You too. Bye.     """

    # Временно подставляем фиктивный meet_id (если БД не требуется)
    meet_id = "078f4a73-c6d6-4654-b991-774bda287304"

    result = await analizer.generate_action_items(transcript_text, meet_id)
    print("\n=== 🧾 Action Items ===\n")
    print(result)

if __name__ == "__main__":
    await main()



=== 🧾 Action Items ===

No action items identified.


In [6]:
 
from src.backend.modules.meetingAnalizer import MeetingAnalizer

async def main():
    analizer = MeetingAnalizer()

    # Пример текста, который ты хочешь проверить
    transcript_text = """
   Unknown: Ну, короче, я просто могу так пообщаться, просто что-то сказать, как бы не вопрос.Unknown: А почему не отработал закон нарушения?Unknown: Ну давай еще раз наружу.Morzh: Давай я тогда скажу.Unknown: Для того, чтобы продолжить нашу работу, оплатите мои услуги в сумме 150% от вашего месячного дохода.Unknown: А чё оно так долго работает?Unknown: Ну, это уже долгое время прошло.Morzh: Это как-то не реал тайм вообще.Unknown: Ну блядь, минуту прошла, чувак.
    """

    # Временно подставляем фиктивный meet_id (если БД не требуется)
    meet_id = "078f4a73-c6d6-4654-b991-774bda287304"

    result = await analizer.generate_action_items(transcript_text, meet_id)
    print("\n=== 🧾 Action Items ===\n")
    print(result)

if __name__ == "__main__":
    await main()


=== 🧾 Action Items ===

No action items identified.


In [1]:
 
from src.backend.modules.meetingAnalizer import MeetingAnalizer

async def main():
    analizer = MeetingAnalizer()

    # Пример текста, который ты хочешь проверить
    transcript_text = """
   Unknown: Ну, короче, я просто могу так пообщаться, просто что-то сказать, как бы не вопрос.Unknown: А почему не отработал закон нарушения?Unknown: Ну давай еще раз наружу.Morzh: Давай я тогда скажу.Unknown: Для того, чтобы продолжить нашу работу, оплатите мои услуги в сумме 150% от вашего месячного дохода.Unknown: А чё оно так долго работает?Unknown: Ну, это уже долгое время прошло.Morzh: Это как-то не реал тайм вообще.Unknown: Ну блядь, минуту прошла, чувак.
    """

    # Временно подставляем фиктивный meet_id (если БД не требуется)
    meet_id = "ad8e4c61-fddb-4b25-8acf-09f32f0e4fdc"

    result = await analizer.generate_action_items(transcript_text, meet_id)
    print("\n=== 🧾 Action Items ===\n")
    print(result)

if __name__ == "__main__":
    await main()


=== 🧾 Action Items ===

No action items identified.
